ez akkor érdekes, ha s0 s1 mérés szétválasztás van

In [123]:

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
from tqdm import tqdm

from scipy.ndimage import gaussian_filter1d
from analysis_utils import compute_respcorr_split_half

Expecting to have:
* Spikes of the neural data to analyze: npy file having array of (n_trials, n_timepoints, n_neurons)
* Cell database: cells_caiman.cellDB_pickle having records for each neuron - this will be extended with the calculated data

In [124]:
#jobFolder_str=r"GBM11\g11_0409_zebra"
jobFolder_str=r"GBM11\g11_0409_zebra5"
jobFolder_str=r"GBM11\g11_0415_zebra5"
jobFolder_str=r"GBM11\g11_0423_zebra6"
jobFolder_str=r"GBM11\g11_0508_full"
jobFolder_str=r"GBM11\g11_0601_full"
jobFolder_str=r"GBM11\g11_0601_regio2"

#jobFolder_str=r"GBM15\g15_0408_zebra"
#jobFolder_str=r"GBM15\g15_0422_zebra3"

In [125]:
spks_path = "D:\\SynologyDriveSyncedDATA\\PROCESSED\\GBM\\" +  jobFolder_str + "\\ZEBRA_ANALYSIS\\resps_all.npy"

temppath = r'D:\SynologyDriveSyncedDATA\PROCESSED\Waven'


if Path(spks_path).exists():   print(spks_path)
else: print(f"File not found: {spks_path}")

D:\SynologyDriveSyncedDATA\PROCESSED\GBM\GBM11\g11_0601_regio2\ZEBRA_ANALYSIS\resps_all.npy


## Load data

In [126]:
spks=np.load(spks_path)
print(f"spks shape: {spks.shape} (n_trials, n_timepoints, n_neurons)")
n_trials=spks.shape[0]
n_timepoints=spks.shape[1]
n_neurons=spks.shape[2]
working_dir = Path(spks_path).parent
print(f"Working directory: {working_dir}")

spks shape: (3, 12600, 2167) (n_trials, n_timepoints, n_neurons)
Working directory: D:\SynologyDriveSyncedDATA\PROCESSED\GBM\GBM11\g11_0601_regio2\ZEBRA_ANALYSIS


In [127]:
shift_samples = 0

target_fps = 30
average_FWHM_sec = 0.33 #sec
smooth_fwhm_samples = int(np.round(target_fps * average_FWHM_sec))

In [128]:
if smooth_fwhm_samples > 0:
    spks = gaussian_filter1d(spks, sigma=smooth_fwhm_samples / 2.355,  axis=1)
if shift_samples != 0:
    spks = np.roll(spks, -shift_samples, axis=1)

In [129]:
# Repeatability
respcorr = compute_respcorr_split_half(spks)

# top 3 most repeatable neurons
top_neurons = np.argsort(respcorr)[-3:]
print(f"Top 3 most repeatable neurons: {top_neurons}")

Computing split-half correlation per neuron: 100%|██████████| 2167/2167 [00:01<00:00, 1795.53it/s]

Top 3 most repeatable neurons: [1671 1708 1703]


## Loading caiman results

In [130]:
#save results into cell database
input_pickle_path= working_dir / "cells_caiman.cellDB_pickle"
if input_pickle_path.exists():
    df_cells=pd.read_pickle(open(input_pickle_path,"rb"))
else:
    df_cells = pd.DataFrame()
    for _idx in range(n_neurons): #handling only good components
            record={}
            record['cell_id'] = _idx
            record['SeriesID'] = 'unknown'
            df_cells = pd.concat([df_cells, pd.DataFrame([record])], ignore_index=True)

df_cells = df_cells.set_index("cell_id", drop=False)

df_cells["RF_indexes"] = pd.Series([None] * len(df_cells), dtype="object")
df_cells["WL_transient_squared"] = pd.Series([None] * len(df_cells), dtype="object")
df_cells["WL_transient_phase"] = pd.Series([None] * len(df_cells), dtype="object")
df_cells["Cell_activity"] = pd.Series([None] * len(df_cells), dtype="object")
df_cells["tun_xs"] = pd.Series([None] * len(df_cells), dtype="object")
df_cells["tun_ys"] = pd.Series([None] * len(df_cells), dtype="object")
df_cells["tun_angles"] = pd.Series([None] * len(df_cells), dtype="object")
df_cells["tun_sizes"] = pd.Series([None] * len(df_cells), dtype="object")
df_cells["tun_freqs"] = pd.Series([None] * len(df_cells), dtype="object")
df_cells["tun_drifts"] = pd.Series([None] * len(df_cells), dtype="object")

for _idx in range(n_neurons):
    df_cells.loc[_idx,'Repeatability'] = respcorr[_idx]



In [131]:
print("Total number of cells:", n_neurons)
print("Accepted cells:", df_cells["Accepted"].sum())
print("Responsive cells:", (df_cells["Repeatability"] > 0.2).sum())
print(
    "Accepted responsive cells:",
    (df_cells["Accepted"] & (df_cells["Repeatability"] > 0.2)).sum()
)

Total number of cells: 2167
Accepted cells: 500
Responsive cells: 765
Accepted responsive cells: 390


In [132]:
print(jobFolder_str)

GBM11\g11_0601_regio2
